# 02 — Make RTN-4bit, GPTQ-4bit (C4) and GPTQ-4bit (math) models (+ reproduce the GPTQ paper)

Settings: Accelerator = **GPU T4 x2**, Internet = **On**. Run with **Save Version → Save & Run All**
so the quantized models are saved (they are needed by notebook 03).

**Part A — Reproduction (OPT-125M).** The GPTQ paper (Table 3) reports WikiText2 perplexity for
OPT-125M at 4 bits (per-row grid): **FP16 27.65, RTN 37.28, GPTQ 31.12**.
If our code gets close to these, our RTN and GPTQ match the paper.

**Part B — Our models (Qwen2.5-1.5B-Instruct).** 4-bit, group size 128, asymmetric grid,
Two GPTQ versions with the same grid and settings, differing only in calibration text:
**C4** (128 × 2,048 web tokens, the paper's setup) and **math** (128 × 2,048 tokens of GSM8K
*training* problems with solutions; the test set is never used). All three models are saved.

**Checkpointing:** each finished step (OPT results, Qwen FP16, RTN model, GPTQ model) is saved
right away. If a run stops, attach that failed version as an Input (**+ Add Input → Your Work**)
and run again: finished steps are skipped.

**Saved files (in `/kaggle/working`, kept in the version's Output tab):**
`qwen2.5-1.5b-rtn4-g128/`, `qwen2.5-1.5b-gptq4-g128-c4/`, `qwen2.5-1.5b-gptq4-g128-math/` (the models),
`02_results.json` (all perplexities), `02_settings.json` (all settings and versions).

Note: the quantized models are stored as FP16 files whose weights take only 2^4 = 16 values per
group ("simulated quantization", as in the GPTQ paper's accuracy experiments). The answers are
exactly those of a real 4-bit model; only speed and file size are not reduced, which this project
does not measure.

In [ ]:
# 1. Versions and GPU
import sys, torch, transformers, datasets
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("GPU         ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on the GPU!")

In [ ]:
# 2. Settings
QWEN_ID   = "Qwen/Qwen2.5-1.5B-Instruct"
OPT_ID    = "facebook/opt-125m"
BITS      = 4
GROUP     = 128      # group size for Qwen (the paper's OPT table uses per-row = -1)
NSAMPLES  = 128      # calibration: 128 random pieces ...
SEQLEN    = 2048     # ... of 2,048 tokens each, from C4
SEED      = 0
DAMP      = 0.01     # 1% dampening
OUT_DIR   = "/kaggle/working"

In [ ]:
# 3. RTN and GPTQ (plain PyTorch, following the original GPTQ code; tested offline)
# RTN and GPTQ in plain PyTorch, following the original GPTQ code
# (IST-DASLab/gptq: quant.py, gptq.py, opt.py). Both methods use the SAME
# Quantizer, so they share exactly the same grid definition.
import math
import torch
import torch.nn as nn


# ---------------------------------------------------------------------------
# 1. The grid (asymmetric min-max, as in Section 2.1 of the survey)
# ---------------------------------------------------------------------------
def quantize(x, scale, zero, maxq):
    """q = clip(round(x/s) + z, 0, 2^b - 1);  w_hat = s * (q - z)"""
    q = torch.clamp(torch.round(x / scale) + zero, 0, maxq)
    return scale * (q - zero)


class Quantizer:
    def __init__(self, bits, sym=False):
        self.maxq = 2 ** bits - 1
        self.sym = sym

    def find_params(self, w):
        """One scale and zero-point per row of w (w = one group of columns)."""
        xmin = torch.minimum(w.min(dim=1).values, torch.zeros(1, device=w.device))
        xmax = torch.maximum(w.max(dim=1).values, torch.zeros(1, device=w.device))
        if self.sym:
            xmax = torch.maximum(xmin.abs(), xmax)
            xmin = -xmax
        both_zero = (xmin == 0) & (xmax == 0)
        xmin[both_zero], xmax[both_zero] = -1, 1
        self.scale = ((xmax - xmin) / self.maxq).unsqueeze(1)
        if self.sym:
            self.zero = torch.full_like(self.scale, (self.maxq + 1) / 2)
        else:
            self.zero = torch.round(-xmin / self.scale.squeeze(1)).unsqueeze(1)

    def __call__(self, x):
        return quantize(x, self.scale, self.zero, self.maxq)


# ---------------------------------------------------------------------------
# 2. RTN: round each group independently, no data
# ---------------------------------------------------------------------------
@torch.no_grad()
def rtn_linear_(layer, bits, groupsize, sym=False):
    W = layer.weight.data.float()
    cols = W.shape[1]
    g = cols if groupsize == -1 else groupsize
    qz = Quantizer(bits, sym)
    Q = torch.empty_like(W)
    for c in range(0, cols, g):
        qz.find_params(W[:, c:c + g])
        Q[:, c:c + g] = qz(W[:, c:c + g])
    layer.weight.data = Q.to(layer.weight.dtype)


# ---------------------------------------------------------------------------
# 3. GPTQ for one linear layer (Algorithm 1 of the paper)
# ---------------------------------------------------------------------------
class GPTQ:
    def __init__(self, layer):
        self.layer = layer
        cols = layer.weight.shape[1]
        self.H = torch.zeros((cols, cols), device=layer.weight.device, dtype=torch.float32)
        self.nsamples = 0

    def add_batch(self, inp):
        """Accumulate H = 2 X X^T / n from the inputs this layer receives."""
        inp = inp.reshape(-1, inp.shape[-1]).t().float()        # (cols, tokens)
        self.H *= self.nsamples / (self.nsamples + 1)
        self.nsamples += 1
        inp = math.sqrt(2 / self.nsamples) * inp
        self.H += inp @ inp.t()

    @torch.no_grad()
    def quantize_(self, bits, groupsize=128, sym=False, percdamp=0.01, blocksize=128):
        W = self.layer.weight.data.clone().float()
        rows, cols = W.shape
        qz = Quantizer(bits, sym)
        if groupsize == -1:
            qz.find_params(W)
        H = self.H
        del self.H
        dead = torch.diag(H) == 0                    # inputs that are always 0
        H[dead, dead] = 1
        W[:, dead] = 0
        damp = percdamp * torch.mean(torch.diag(H))  # 1% dampening
        H += damp * torch.eye(cols, device=H.device)
        # Cholesky form of H^-1 (upper triangular), as in the paper
        H = torch.linalg.cholesky(H)
        H = torch.cholesky_inverse(H)
        Hinv = torch.linalg.cholesky(H, upper=True)

        Q = torch.zeros_like(W)
        losses = torch.zeros(rows, device=W.device)
        for i1 in range(0, cols, blocksize):          # lazy batch updates
            i2 = min(i1 + blocksize, cols)
            W1 = W[:, i1:i2].clone()
            Q1 = torch.zeros_like(W1)
            Err1 = torch.zeros_like(W1)
            Hinv1 = Hinv[i1:i2, i1:i2]
            for i in range(i2 - i1):                  # fixed column order
                if groupsize != -1 and (i1 + i) % groupsize == 0:
                    qz.find_params(W[:, i1 + i:i1 + i + groupsize])
                w = W1[:, i]
                d = Hinv1[i, i]
                q = qz(w.unsqueeze(1)).flatten()
                Q1[:, i] = q
                losses += (w - q) ** 2 / d ** 2 / 2
                err = (w - q) / d
                W1[:, i:] -= err.unsqueeze(1) @ Hinv1[i, i:].unsqueeze(0)
                Err1[:, i] = err
            Q[:, i1:i2] = Q1
            W[:, i2:] -= Err1 @ Hinv[i1:i2, i2:]
        self.layer.weight.data = Q.to(self.layer.weight.dtype)
        return losses.sum().item()


# ---------------------------------------------------------------------------
# 4. Whole-model helpers
# ---------------------------------------------------------------------------
def get_layers(model):
    """Transformer blocks for Qwen/Llama-style and OPT-style models."""
    inner = model.model
    if hasattr(inner, "layers"):
        return inner.layers
    if hasattr(inner, "decoder") and hasattr(inner.decoder, "layers"):
        return inner.decoder.layers
    raise ValueError("Unknown architecture")


def block_linears(block):
    return {n: m for n, m in block.named_modules() if isinstance(m, nn.Linear)}


@torch.no_grad()
def rtn_model_(model, bits=4, groupsize=128, sym=False):
    for block in get_layers(model):
        for m in block_linears(block).values():
            rtn_linear_(m, bits, groupsize, sym)


class _Catcher(nn.Module):
    """Stands in for block 0 to record its inputs, then stops the forward pass."""
    def __init__(self, m, store):
        super().__init__()
        self.m = m
        self.store = store

    def __getattr__(self, name):          # pass through attributes like attention_type
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.m, name)

    def forward(self, *args, **kwargs):
        hs = args[0] if args else kwargs.pop("hidden_states")
        self.store["inps"].append(hs)
        if self.store["kwargs"] is None:
            self.store["kwargs"] = kwargs
        raise ValueError("captured")


def _first(out):
    return out[0] if isinstance(out, (tuple, list)) else out


@torch.no_grad()
def gptq_model_(model, calib, bits=4, groupsize=128, sym=False,
                percdamp=0.01, blocksize=128, verbose=True):
    """Quantize block by block. Each block is calibrated on the outputs of the
    blocks before it that are ALREADY quantized (as in the GPTQ paper)."""
    layers = get_layers(model)
    dev = next(model.parameters()).device
    store = {"inps": [], "kwargs": None}
    layers[0] = _Catcher(layers[0], store)
    for x in calib:
        try:
            model(x.to(dev), use_cache=False)
        except ValueError:
            pass
    layers[0] = layers[0].m
    inps = torch.cat(store["inps"], dim=0)
    kw = store["kwargs"]
    outs = torch.zeros_like(inps)

    for li, block in enumerate(layers):
        lins = block_linears(block)
        gptq = {n: GPTQ(m) for n, m in lins.items()}
        hooks = [m.register_forward_hook(
                    lambda mod, inp, out, n=n: gptq[n].add_batch(inp[0].data))
                 for n, m in lins.items()]
        for j in range(inps.shape[0]):
            block(inps[j:j + 1], **kw)
        for h in hooks:
            h.remove()
        err = {}
        for n in lins:
            err[n] = gptq[n].quantize_(bits, groupsize, sym, percdamp, blocksize)
            del gptq[n]
        for j in range(inps.shape[0]):           # outputs of the QUANTIZED block
            outs[j] = _first(block(inps[j:j + 1], **kw))[0]
        inps, outs = outs, inps
        if verbose:
            print(f"  block {li:2d} done | total error {sum(err.values()):.1f}")
        torch.cuda.empty_cache() if torch.cuda.is_available() else None


# ---------------------------------------------------------------------------
# 5. Check: every (row, group) uses at most 2^bits distinct values
# ---------------------------------------------------------------------------
@torch.no_grad()
def check_grid(model, bits, groupsize):
    worst = 0
    for block in get_layers(model):
        for m in block_linears(block).values():
            W = m.weight.data.float()
            g = W.shape[1] if groupsize == -1 else groupsize
            for c in range(0, W.shape[1], g):
                grp = W[:, c:c + g]
                worst = max(worst, max(len(torch.unique(r)) for r in grp[:8]))
    return worst

In [ ]:
# 4. Data: C4 calibration samples and WikiText2 perplexity
import random, gc, json, time, re
import torch.nn.functional as F
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

c4 = load_dataset("allenai/c4", data_files={"train": "en/c4-train.00000-of-01024.json.gz"},
                  split="train")
wiki_text = "\n\n".join(load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1",
                                      split="test")["text"])

def get_c4_calib(tok, n=NSAMPLES, seqlen=SEQLEN, seed=SEED):
    # Same procedure as the GPTQ code: random document, random 2,048-token window
    random.seed(seed)
    samples = []
    while len(samples) < n:
        ids = tok(c4[random.randint(0, len(c4) - 1)]["text"], return_tensors="pt").input_ids
        if ids.shape[1] <= seqlen:
            continue
        i = random.randint(0, ids.shape[1] - seqlen - 1)
        samples.append(ids[:, i:i + seqlen])
    return samples

gsm_train = load_dataset("openai/gsm8k", "main", split="train")   # TRAIN split only (no test leakage)

def gsm8k_text(ex):
    # plain text, same style for every example; calculator notes <<...>> removed
    sol = re.sub(r"<<[^>]*>>", "", ex["answer"])
    sol = re.sub(r"####\s*(.+)$", r"The answer is \1.", sol.strip())
    return "Question: " + ex["question"] + "\nAnswer: " + sol

def get_math_calib(tok, n=NSAMPLES, seqlen=SEQLEN, seed=SEED):
    # Same shape as C4 calibration (n pieces of seqlen tokens), but the text is
    # GSM8K training problems with their worked solutions, joined into one stream.
    order = list(range(len(gsm_train)))
    random.Random(seed).shuffle(order)
    text = "\n\n".join(gsm8k_text(gsm_train[i]) for i in order)
    ids = tok(text, return_tensors="pt").input_ids
    rng = random.Random(seed)
    return [ids[:, i:i + seqlen] for i in
            (rng.randint(0, ids.shape[1] - seqlen - 1) for _ in range(n))]

@torch.no_grad()
def ppl_wikitext2(model, tok, seqlen=SEQLEN):
    ids = tok(wiki_text, return_tensors="pt").input_ids
    n = ids.shape[1] // seqlen
    nll = 0.0
    for i in range(n):
        x = ids[:, i * seqlen:(i + 1) * seqlen].to(model.device)
        logits = model(x, use_cache=False).logits[:, :-1, :].float()
        nll += F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                               x[:, 1:].reshape(-1)).item() * seqlen
    return math.exp(nll / (n * seqlen))

def load(model_id):
    m = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16,
                                             device_map={"": 0})
    return m.eval()

def free():
    gc.collect(); torch.cuda.empty_cache()

# ---- checkpointing: every finished step is saved; a re-run skips finished steps ----
import os, glob, shutil
RESULTS_FILE = "02_results.json"
RTN_NAME  = f"qwen2.5-1.5b-rtn{BITS}-g{GROUP}"
GPTQ_NAME = f"qwen2.5-1.5b-gptq{BITS}-g{GROUP}-c4"
GPTQM_NAME = f"qwen2.5-1.5b-gptq{BITS}-g{GROUP}-math"

def restore(name):
    """Copy `name` from an attached earlier run (/kaggle/input) if it is not here yet."""
    dst = os.path.join(OUT_DIR, name)
    if os.path.exists(dst):
        return True
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits:
        return False
    (shutil.copytree if os.path.isdir(hits[0]) else shutil.copy)(hits[0], dst)
    print("restored", name, "from", hits[0])
    return True

def save_results():
    with open(os.path.join(OUT_DIR, RESULTS_FILE), "w") as f:
        json.dump(results, f, indent=1)

results = {}
if restore(RESULTS_FILE):
    results = json.load(open(os.path.join(OUT_DIR, RESULTS_FILE)))
    print("already finished:", list(results))

## Part A — Reproduce the GPTQ paper on OPT-125M (4-bit, per-row grid)

In [ ]:
# 5. OPT-125M: FP16 vs RTN vs GPTQ  (paper: 27.65 / 37.28 / 31.12)
t0 = time.time()
opt_tok = AutoTokenizer.from_pretrained(OPT_ID)
if "opt125m_fp16" not in results:
    m = load(OPT_ID); results["opt125m_fp16"] = ppl_wikitext2(m, opt_tok); del m; free()
    save_results()
if "opt125m_rtn4" not in results:
    m = load(OPT_ID); rtn_model_(m, BITS, groupsize=-1)
    results["opt125m_rtn4"] = ppl_wikitext2(m, opt_tok); del m; free()
    save_results()
if "opt125m_gptq4" not in results:
    opt_calib = get_c4_calib(opt_tok)
    m = load(OPT_ID); gptq_model_(m, opt_calib, BITS, groupsize=-1, percdamp=DAMP, verbose=False)
    results["opt125m_gptq4"] = ppl_wikitext2(m, opt_tok); del m; free()
    save_results()

print(f"{'':10s}{'ours':>8s}{'paper':>8s}")
for k, paper in [("fp16", 27.65), ("rtn4", 37.28), ("gptq4", 31.12)]:
    print(f"{k:10s}{results['opt125m_' + k]:8.2f}{paper:8.2f}")
print(f"({time.time() - t0:.0f} s)")

## Part B — Qwen2.5-1.5B-Instruct: FP16, RTN-4bit, GPTQ-4bit (group size 128)

In [ ]:
# 6. Qwen FP16 baseline perplexity
tok = AutoTokenizer.from_pretrained(QWEN_ID)
if "qwen_fp16" not in results:
    m = load(QWEN_ID)
    results["qwen_fp16"] = ppl_wikitext2(m, tok)
    del m; free(); save_results()
print(f"Qwen FP16 WikiText2 perplexity: {results['qwen_fp16']:.3f}")

In [ ]:
# 7. Qwen RTN-4bit (g128): quantize, check, save
t0 = time.time()
RTN_DIR = f"{OUT_DIR}/{RTN_NAME}"
if "qwen_rtn4" in results and restore(RTN_NAME):
    print("RTN model already done -> skipped")
else:
    m = load(QWEN_ID)
    rtn_model_(m, BITS, GROUP)
    print(f"max distinct values per group: {check_grid(m, BITS, GROUP)} (must be <= {2**BITS})")
    m.save_pretrained(RTN_DIR); tok.save_pretrained(RTN_DIR)
    print("saved", RTN_DIR)
    results["qwen_rtn4"] = ppl_wikitext2(m, tok)
    del m; free(); save_results()
print(f"RTN-4bit perplexity: {results['qwen_rtn4']:.3f} | {time.time()-t0:.0f} s")

In [ ]:
# 8. Qwen GPTQ-4bit (g128, C4 calibration): quantize, check, save  (~10-20 min)
t0 = time.time()
GPTQ_DIR = f"{OUT_DIR}/{GPTQ_NAME}"
if "qwen_gptq4" in results and restore(GPTQ_NAME):
    print("GPTQ model already done -> skipped")
else:
    qwen_calib = get_c4_calib(tok)
    m = load(QWEN_ID)
    gptq_model_(m, qwen_calib, BITS, GROUP, percdamp=DAMP)
    print(f"max distinct values per group: {check_grid(m, BITS, GROUP)} (must be <= {2**BITS})")
    m.save_pretrained(GPTQ_DIR); tok.save_pretrained(GPTQ_DIR)
    print("saved", GPTQ_DIR)
    results["qwen_gptq4"] = ppl_wikitext2(m, tok)
    del m; free(); save_results()
print(f"GPTQ-4bit perplexity: {results['qwen_gptq4']:.3f} | {time.time()-t0:.0f} s")

In [ ]:
# 8b. Qwen GPTQ-4bit calibrated on MATH (GSM8K train): same grid and settings,
#     only the calibration text changes  (~10-20 min)
t0 = time.time()
GPTQM_DIR = f"{OUT_DIR}/{GPTQM_NAME}"
if "qwen_gptq4_math" in results and restore(GPTQM_NAME):
    print("GPTQ-math model already done -> skipped")
else:
    math_calib = get_math_calib(tok)
    print("example calibration text:", tok.decode(math_calib[0][0, :80]))
    m = load(QWEN_ID)
    gptq_model_(m, math_calib, BITS, GROUP, percdamp=DAMP)
    print(f"max distinct values per group: {check_grid(m, BITS, GROUP)} (must be <= {2**BITS})")
    m.save_pretrained(GPTQM_DIR); tok.save_pretrained(GPTQM_DIR)
    print("saved", GPTQM_DIR)
    results["qwen_gptq4_math"] = ppl_wikitext2(m, tok)
    del m; free(); save_results()
print(f"GPTQ-4bit (math calib) perplexity: {results['qwen_gptq4_math']:.3f} | {time.time()-t0:.0f} s")

In [ ]:
# 9. Summary + save settings and results
print("WikiText2 perplexity (lower is better)")
print(f"  OPT-125M  FP16 {results['opt125m_fp16']:.2f} | RTN-4 {results['opt125m_rtn4']:.2f} | "
      f"GPTQ-4 {results['opt125m_gptq4']:.2f}   (paper: 27.65 | 37.28 | 31.12)")
print(f"  Qwen-1.5B FP16 {results['qwen_fp16']:.2f} | RTN-4 {results['qwen_rtn4']:.2f} | "
      f"GPTQ-4 C4 {results['qwen_gptq4']:.2f} | GPTQ-4 math {results['qwen_gptq4_math']:.2f}")
settings = dict(qwen=QWEN_ID, opt=OPT_ID, bits=BITS, group=GROUP, sym=False,
                calib={"gptq4": "C4 en shard 00000", "gptq4_math": "GSM8K train (question + solution)"}, nsamples=NSAMPLES, seqlen=SEQLEN, seed=SEED,
                damp=DAMP, blocksize=128, torch=torch.__version__,
                transformers=transformers.__version__)
with open(f"{OUT_DIR}/02_settings.json", "w") as f:
    json.dump(settings, f, indent=1)
print("saved 02_results.json (perplexities) and 02_settings.json")

## After it finishes
1. This must run as **Save Version → Save & Run All (Commit)** so the two model folders are kept.
2. Send the output of **cells 5, 7, 8 and 9**.
3. Notebook 03 will load these folders by adding this notebook's output as an **Input**.